# Notebook 05 — Predicția afinității proteină-ligand

Ai parcurs Julia (nb 01), hypergraphuri + ODE (nb 02), fundamentele machine learning (nb 03) și Hypergraph Neural Networks (nb 04). Acum facem ce face *de fapt* un grup de cercetare în ML aplicat la proteine:

**descărcăm date reale → analizăm → antrenăm un model → evaluăm pe set de test.**

**Tema:** predicția **afinității de legare proteină-ligand**. Dat un complex proteină-medicament, cât de tare se leagă cei doi? Această sarcină stă la baza descoperirii de medicamente: înainte să sintetizezi 1000 de compuși și să-i testezi în laborator, prezici care dintre ei vor avea afinitate bună — economisești timp și bani.

**Sursa de date:** [LP-PDBBind](https://github.com/THGLab/LP-PDBBind) — un mirror curat și public al [PDBbind v2020](http://www.pdbbind.org.cn/), cu **~19 400 complexe proteină-ligand** cu afinități măsurate experimental ($K_d$ sau $K_i$). Include SMILES, secvențe, rezoluție și un **split *leak-proof*** train/val/test [Li et al., 2023].

**De ce LP-PDBBind și nu PDBbind direct?**
- PDBbind oficial cere înregistrare; LP-PDBBind e un derivat public, științific, publicat în [JCIM 2023](https://arxiv.org/abs/2308.09639).
- Adaugă un **split leak-proof**: proteine cu secvențe similare (>30% identitate) sunt forțate în același split. Asta evită ca modelul să "învețe" o secvență din train și apoi s-o regăsească ușor în test — un fenomen real care a umflat artificial multe rezultate publicate.
- Include și **SMILES-ul ligandului pre-extras** (nu trebuie să parsăm fișiere PDB separat).

**Parcurs:**
1. **Setup** — pachete noi: `MolecularGraph.jl`, `GLM.jl`, `CSV.jl`, `DataFrames.jl`.
2. **Descărcare date** — un singur CSV de pe GitHub (~10 MB).
3. **Analiză exploratorie (EDA)** — distribuții, splits, ce avem în mâini.
4. **Featurizare** — transformăm SMILES în descriptori numerici cu `MolecularGraph.jl`.
5. **Baseline** — regresie liniară cu `GLM.jl`.
6. **MLP** — rețea neuronală cu `Flux.jl`.
7. **Evaluare** — MAE, RMSE, R², parity plot, residuals.
8. **Vizualizare 3D** — un complex prezis bine vs. unul ratat.
9. **Discuție** — limitări + ce ar urma (HGNN din nb 04!).

> 💡 Acest notebook e **end-to-end**: la final ai un pipeline complet de cercetare ML aplicată. E exact tipul de săptămână 1 la UCD.

## Cap. 1 — Setup

Pachete noi peste cele din notebook-urile precedente:

- `Downloads` — descarcă fișiere prin HTTP (stdlib, vine cu Julia)
- `CSV`, `DataFrames` — citire + manipulare tabele
- `StatsBase`, `Statistics` — statistici
- `MolecularGraph` — parser SMILES + descriptori moleculari (pur Julia)
- `GLM` — regresie liniară (baseline)
- `Flux` — deja folosit în nb 03 (rețele neuronale)
- `Bio3DView` — deja folosit în nb 04 (vizualizare 3D)

Comenzile de instalare sunt mai jos. Le rulezi o singură dată; după ce-au mers, comentează-le ca să nu mai pierzi timp la fiecare repornire.

In [1]:
using Pkg
for pkg in ["CSV", "DataFrames", "StatsBase", "Plots", "StatsPlots",
            "MolecularGraph", "GLM", "Flux", "Bio3DView"]
    Pkg.add(pkg; io=devnull)
end
println("✓ Pachetele sunt instalate.")

✓ Pachetele sunt instalate.


In [ ]:
using Downloads, CSV, DataFrames, Dates, Statistics, StatsBase, Random
using Plots, StatsPlots
using MolecularGraph
using GLM
using Flux
using Bio3DView
using LinearAlgebra

Random.seed!(42)
println("✓ Pachetele sunt încărcate.")

## Cap. 2 — Descărcare date

Descărcăm CSV-ul LP-PDBBind o singură dată local, în folderul `../data/`. Dacă există deja, sărim peste — economisește bandă și e politicos cu serverul GitHub.

**Conținut CSV:** ~19 400 rânduri, ~10 MB. Coloanele importante pentru noi:

| Coloană | Conținut |
|---|---|
| `header` | codul PDB (ex: `6r8o`) |
| `smiles` | SMILES-ul ligandului |
| `seq` | secvența proteinei (1-letter aa) |
| `resolution` | rezoluția cristalului (Å) — mai mic = mai precis |
| `date` | data depunerii structurii în PDB |
| `type` | clasa funcțională (kinaze, proteaze, ...) |
| `new_split` | `"train"`, `"val"`, `"test"` (split-ul LP-PDBBind leak-proof) |
| `value` | **target-ul** = $-\log_{10}(K_d)$ sau $-\log_{10}(K_i)$ în molar |
| `covalent` | `true` dacă ligandul leagă covalent (îi excludem) |

**Interpretarea `value`:** scala $-\log K$ e logaritmică:

| $K_d$ | `value` | Interpretare |
|---|---|---|
| 1 nM ($10^{-9}$ M) | 9.0 | legare foarte tare (drug aprobat) |
| 1 μM ($10^{-6}$ M) | 6.0 | legare moderată |
| 1 mM ($10^{-3}$ M) | 3.0 | legare slabă |

Range tipic în dataset: 2 – 12.

In [ ]:
const DATA_DIR = joinpath(@__DIR__, "..", "data")
const LP_PDBBIND_URL = "https://raw.githubusercontent.com/THGLab/LP-PDBBind/master/dataset/LP_PDBBind.csv"
const LP_PDBBIND_PATH = joinpath(DATA_DIR, "LP_PDBBind.csv")

isdir(DATA_DIR) || mkpath(DATA_DIR)

if isfile(LP_PDBBIND_PATH)
    println("✓ Datele există deja la: ", LP_PDBBIND_PATH)
    println("  (Dacă vrei să forțezi re-download, șterge fișierul.)")
else
    println("⏳ Descarc LP-PDBBind (~10 MB)...")
    Downloads.download(LP_PDBBIND_URL, LP_PDBBIND_PATH)
    println("✓ Descărcat la: ", LP_PDBBIND_PATH)
end

println("\nMărime fișier: ", round(filesize(LP_PDBBIND_PATH) / 1024^2, digits=2), " MB")

### Citire în DataFrame

`CSV.read` parsează CSV-ul direct într-un `DataFrame` (pachetul `DataFrames.jl`, similar cu `pandas` din alte ecosisteme). Argumentul `missingstring=["(none)", ""]` îi spune că aceste valori reprezintă date lipsă, nu string-uri reale.

In [ ]:
df_raw = CSV.read(LP_PDBBIND_PATH, DataFrame; missingstring=["(none)", ""])
println("Înregistrări: ", nrow(df_raw))
println("Coloane: ", ncol(df_raw))
println()
println("Tipuri coloane:")
for c in names(df_raw)
    println("  - $c  ::  $(eltype(df_raw[!, c]))")
end

In [ ]:
first(df_raw, 5)

### Curățare

Pentru ML avem nevoie de rânduri **complete** și **utilizabile**:
- `smiles` și `value` să existe (target și input principal),
- `new_split` să fie unul din `train`/`val`/`test`,
- ligand **non-covalent** (covalenții sunt un caz special, lângă chimie organică reactivă; modelele standard nu îi tratează bine).

In [ ]:
df = filter(row ->
    !ismissing(row.smiles) && !ismissing(row.value) &&
    !ismissing(row.new_split) && row.new_split in ("train", "val", "test") &&
    !ismissing(row.covalent) && row.covalent == false,
    df_raw)

println("După curățare: $(nrow(df)) înregistrări (din $(nrow(df_raw)))")
println()
println("Distribuție split-uri:")
for split in ("train", "val", "test")
    n = sum(df.new_split .== split)
    pct = round(100 * n / nrow(df), digits=1)
    println("  $split : $n  ($pct%)")
end

## Cap. 3 — Analiză exploratorie a datelor (EDA)

Înainte de orice model, **trebuie** să te uiți la date. EDA = "ce e acolo, ce distribuție au lucrurile, ce corelații există, ce e ciudat". Cox (2005): *"Most statistical disasters originate in poor data understanding."*

Întrebări pe care le punem:

1. Cum arată distribuția target-ului `value`? Este normală? Are outliers?
2. Cât de echilibrate sunt split-urile?
3. Rezoluția cristalului influențează afinitatea măsurată?
4. Există tendințe temporale (structurile vechi diferă de cele noi)?
5. Care sunt clasele de proteine cele mai frecvente?

### 3.1 Distribuția target-ului

In [ ]:
histogram(df.value, bins=50,
    xlabel="pKd / pKi  (= -log₁₀(K_d) sau -log₁₀(K_i))",
    ylabel="Număr complexe",
    title="Distribuția afinităților de legare în LP-PDBBind",
    label="all data", legend=:topright, color=:steelblue, alpha=0.8,
    size=(750, 420))
vline!([mean(df.value)], color=:red, lw=2,
       label="Medie = $(round(mean(df.value), digits=2))")
vline!([median(df.value)], color=:darkgreen, lw=2, linestyle=:dash,
       label="Mediană = $(round(median(df.value), digits=2))")

**Observații:**
- Distribuția e aproximativ normală, centrată în jurul lui 6 (corespunde unui $K_d$ ~ μM — afinități "tipice" pentru screening farmaceutic).
- Cele mai multe complexe au valori între 4–10 (legare slabă-moderată-puternică).
- Cozi: foarte puține complexe sub 3 sau peste 11 — limita experimentală a tehnicilor de măsurare. E greu să măsori cu precizie legături foarte slabe (concentrații foarte mari de ligand) sau foarte puternice (concentrații sub limita de detecție).

### 3.2 Distribuția pe splits

In [ ]:
split_counts = combine(groupby(df, :new_split), nrow => :count)
sort!(split_counts, :new_split)

bar(split_counts.new_split, split_counts.count,
    xlabel="Split", ylabel="Număr complexe",
    title="Distribuția train/val/test în LP-PDBBind",
    color=[:steelblue, :orange, :forestgreen], legend=false,
    size=(650, 420))
for (i, row) in enumerate(eachrow(split_counts))
    annotate!(i, row.count + 100, text("$(row.count)", 10, :center))
end
plot!()

**Split-ul ~80/10/10** e standard pentru dataset-uri de această dimensiune. Important: aici split-ul **NU e random** — e construit din LP-PDBBind ca să fie *leak-proof*. Două complexe cu proteine având >30% identitate de secvență sunt forțate în același split. Asta dă o estimare onestă a cât de bine generalizează modelul tău la **proteine nemaivăzute**, nu doar la perechi specifice proteină-ligand din distribuția training.

### 3.3 Rezoluție vs. afinitate

In [ ]:
df_res = filter(row -> !ismissing(row.resolution), df)
scatter(df_res.resolution, df_res.value,
    xlabel="Rezoluție cristal (Å)", ylabel="pKd / pKi",
    title="Rezoluție cristalografică vs. afinitate (n=$(nrow(df_res)))",
    markersize=2, alpha=0.25, color=:steelblue, legend=false,
    size=(750, 420))

# Trend liniar pe top
res_clean = collect(skipmissing(df_res.resolution))
val_clean = df_res.value[.!ismissing.(df_res.resolution)]
c = cor(res_clean, val_clean)
annotate!(maximum(res_clean) * 0.85, maximum(val_clean) * 0.95,
          text("Pearson r = $(round(c, digits=3))", 11, :left))

**Observație:** corelația rezoluție–afinitate e foarte slabă (|r| < 0.2). Structurile de rezoluție mare (≤ 1.5 Å) acoperă toată gama de afinități, la fel ca cele de rezoluție mediocră (~ 3 Å). De ce? Pentru că rezoluția e o **proprietate a experimentului cristalografic**, nu o proprietate a interacțiunii proteină-ligand.

O includem totuși ca feature — nu ne strică, și modelul poate folosi inferențial: structurile foarte slabe au valori `value` mai puțin de încredere.

### 3.4 Tendințe temporale

In [ ]:
function parse_year(d)
    ismissing(d) && return missing
    d isa Date && return year(d)
    try
        return year(Date(string(d)))
    catch
        return missing
    end
end

df.year = parse_year.(df.date)
df_dated = filter(row -> !ismissing(row.year), df)

year_stats = combine(groupby(df_dated, :year),
                     :value => mean => :mean_pK,
                     nrow => :count)
sort!(year_stats, :year)

p1 = bar(year_stats.year, year_stats.count,
         xlabel="An", ylabel="Număr structuri",
         title="Structuri depuse pe an",
         color=:steelblue, legend=false)

p2 = plot(year_stats.year, year_stats.mean_pK,
          xlabel="An", ylabel="Afinitate medie",
          title="Afinitate medie pe an",
          color=:red, marker=:circle, lw=2, legend=false)

plot(p1, p2, layout=(2, 1), size=(800, 600))

**Observații:**
- Numărul structurilor pe an a explodat — boom cristalografic + criomicroscopie electronică în ultimul deceniu.
- Afinitatea medie nu prezintă tendințe puternice — chimia farmaceutică nu produce *în medie* drug-uri mai bune, doar mai multe. Există potențial un drift mic spre afinități mai mari (mai multe drug-uri în target portfoliu) dar zgomotos.

### 3.5 Clase de proteine

In [ ]:
type_counts = combine(groupby(df, :type), nrow => :count)
sort!(type_counts, :count, rev=true)
top10 = first(type_counts, 10)

bar(top10.type, top10.count,
    xlabel="Clasă funcțională", ylabel="Număr complexe",
    title="Top 10 clase de proteine în LP-PDBBind",
    xrotation=45, color=:purple, alpha=0.7, legend=false,
    size=(850, 500), bottom_margin=18Plots.mm)

**Observație importantă:** dataset-ul e dominat de **hidrolaze, transferaze și oxidoreductaze** — clasele cele mai studiate în drug discovery (kinaze, proteaze, etc.). Asta înseamnă că:
- Modelul nostru va generaliza **bine pe ținte de aceste tipuri**.
- Va generaliza **prost pe ținte rare** — GPCR, canale ionice, intrinsec dezordonate (IDP).
- Acest bias e o limitare structurală a câmpului ML-pe-proteine, nu o problemă a metodelor noastre.

> 💡 **Mesaj pentru UCD:** când prezinți rezultate pe PDBbind, fii sinceră despre acest bias. Rezultate "state-of-the-art" pe PDBbind nu se traduc 1:1 la GPCR-uri sau ținte virale noi.

## Cap. 4 — Featurizare moleculară

Avem SMILES-uri ca string-uri (`"CC(=O)Oc1ccccc1C(=O)O"` = aspirina). Modelele ML lucrează cu **vectori de numere**. Trebuie să transformăm fiecare SMILES într-un vector $\mathbf{x} \in \mathbb{R}^d$.

Folosim `MolecularGraph.jl` — un pachet **pur Julia** care:
1. Parsează SMILES → graf molecular (atomi + legături + arii valență, aromaticitate, stereochimie).
2. Calculează zeci de **descriptori moleculari standard**.

### Descriptorii pe care îi extragem

| Descriptor | Notație | Ce înseamnă | De ce contează |
|---|---|---|---|
| `mw` | $MW$ | masa moleculară (g/mol) | Lipinski's Ro5: drug-like $\leq 500$ Da |
| `logp` | $\log P$ | hidrofobicitate (Wildman-Crippen) | leagă-se prin buzunare hidrofobe? |
| `hbd` | HBD | număr donori legături H | interacțiuni cu reziduuri acceptoare |
| `hba` | HBA | număr acceptori legături H | interacțiuni cu reziduuri donoare |
| `rotors` | $n_{\text{rot}}$ | legături rotabile | flexibilitate conformațională |
| `heavy` | $n_{\text{heavy}}$ | atomi non-H | mărime efectivă |
| `rings` | $n_{\text{ring}}$ | număr total inele (SSSR) | rigiditate |
| `arom_rings` | $n_{\text{arom}}$ | inele aromatice | stacking π-π cu Phe, Tyr, Trp, His |

Cei 8 descriptori + rezoluția cristalului = **9 features** per complex.

> 📚 **Context:** featurizarea asta e cunoscută ca *physicochemical descriptors* sau *Lipinski-style features* [Lipinski et al., 2001]. E baseline-ul cel mai simplu. Modelele state-of-the-art (EquiBind, DiffDock) folosesc **graful molecular direct** prin GNN-uri sau structura 3D direct prin echivariant networks. Vom discuta la final.

In [ ]:
function descriptors(smiles::AbstractString)
    mol = smilestomol(smiles)
    return (
        mw            = standard_weight(mol),
        logp          = wclogp(mol),
        hbd           = hydrogen_donor_count(mol),
        hba           = hydrogen_acceptor_count(mol),
        rotors        = rotatable_count(mol),
        heavy         = heavy_atom_count(mol),
        rings         = length(sssr(mol)),
        arom_rings    = count(is_ring_aromatic(mol)),
    )
end

# Test pe trei molecule cunoscute
println("🧪 Aspirina (analgezic):")
display(descriptors("CC(=O)Oc1ccccc1C(=O)O"))

println("\n🧪 Cofeina (stimulant):")
display(descriptors("Cn1cnc2c1c(=O)n(C)c(=O)n2C"))

println("\n🧪 Imatinib (Gleevec, anti-cancer):")
display(descriptors("Cc1ccc(NC(=O)c2ccc(CN3CCN(C)CC3)cc2)cc1Nc1nccc(-c2cccnc2)n1"))

**Verificare:**
- **Aspirina** ($C_9H_8O_4$): MW ≈ 180 g/mol, LogP ≈ 1.2 (slab hidrofob), 1 HBD (OH-ul COOH), 4 HBA (cei 4 O), 0 inele aromatice rotabile.
- **Cofeina**: MW ≈ 194, două inele azotate condensate (purine), 0 HBD.
- **Imatinib**: moleculă mult mai mare (MW ≈ 494, la limita Lipinski!) — un drug clasic.

Valorile pot diferi puțin în funcție de versiunea MolecularGraph.jl, dar magnitudinea trebuie să fie corectă.

### Featurizare batch

Calculăm descriptorii pentru toate moleculele din dataset. Folosim un `try/catch` pentru moleculele cu SMILES patologice (rare, dar există).

In [ ]:
function featurize_dataset(df)
    n = nrow(df)
    println("⏳ Calculez descriptori pentru $n molecule...")

    features = Vector{NamedTuple}(undef, n)
    failed_idx = Int[]

    for (i, smiles) in enumerate(df.smiles)
        try
            features[i] = descriptors(smiles)
        catch e
            push!(failed_idx, i)
            features[i] = (mw=NaN, logp=NaN, hbd=-1, hba=-1,
                           rotors=-1, heavy=-1, rings=-1, arom_rings=-1)
        end

        if i % 2000 == 0
            println("  $i / $n  ($(round(100*i/n, digits=1))%)")
        end
    end

    println("✓ Gata. Eșuat la $(length(failed_idx)) molecule din $n.")
    return features, failed_idx
end

features, failed = featurize_dataset(df);

In [ ]:
# Adăugăm features ca coloane în df
df.mw         = [f.mw         for f in features]
df.logp       = [f.logp       for f in features]
df.hbd        = [f.hbd        for f in features]
df.hba        = [f.hba        for f in features]
df.rotors     = [f.rotors     for f in features]
df.heavy      = [f.heavy      for f in features]
df.rings      = [f.rings      for f in features]
df.arom_rings = [f.arom_rings for f in features]

# Reținem doar rândurile cu featurizare completă + rezoluție
df_feat = filter(row -> !isnan(row.mw) && !ismissing(row.resolution), df)
println("Set final pentru modelare: $(nrow(df_feat)) complexe (din $(nrow(df)))")

### Corelații features ↔ target

In [ ]:
feature_cols = [:mw, :logp, :hbd, :hba, :rotors, :heavy, :rings, :arom_rings, :resolution]
X_all = Matrix(df_feat[!, feature_cols])
y_all = df_feat.value

correlations = [cor(X_all[:, i], y_all) for i in 1:length(feature_cols)]

bar(string.(feature_cols), correlations,
    xlabel="Descriptor", ylabel="Corelație Pearson cu pKd/pKi",
    title="Corelația descriptorilor moleculari cu target-ul",
    color=[c > 0 ? :forestgreen : :crimson for c in correlations],
    xrotation=30, legend=false, size=(850, 450),
    bottom_margin=12Plots.mm)
hline!([0], color=:black, lw=1, linestyle=:dash)

println("Corelații Pearson:")
for (col, c) in zip(feature_cols, correlations)
    println("  $col\t: $(round(c, digits=3))")
end

**Observație importantă:** corelațiile individuale sunt **slabe** (toate sub 0.3 în valoare absolută). Asta e tipic în ML aplicat la chimie/biologie: **niciun descriptor singur nu spune mare lucru**, dar **combinații neliniare** ale lor pot fi predictive. De aceea avem nevoie de modele neliniare ca MLP — regresia liniară nu va captura interacțiunile între features.

Descriptorul cu cea mai mare corelație e probabil `heavy` (numărul de atomi grei) — moleculele mai mari tind să aibă **mai multe contacte favorabile** cu proteina (efect cunoscut ca *"ligand efficiency"* — autorii fac corecție pentru asta când compară drug-uri reali).

### Heatmap features-uri sortate

In [ ]:
# Sortăm după target și luăm o subdivizare pentru vizibilitate
sort_idx = sortperm(y_all)
n_show = 200
step = max(1, length(sort_idx) ÷ n_show)
selected = sort_idx[1:step:length(sort_idx)][1:n_show]
X_sorted = X_all[selected, :]

# Standardizare pe coloane pentru vizibilitate
X_norm = (X_sorted .- mean(X_sorted, dims=1)) ./ std(X_sorted, dims=1)

heatmap(X_norm,
    xlabel="Descriptor", ylabel="Moleculă (top→bottom = pKd ascendent)",
    xticks=(1:length(feature_cols), string.(feature_cols)),
    title="Features moleculari (200 molecule sortate după afinitate)",
    c=:RdBu, clim=(-3, 3), size=(800, 600))

**Cum citești heatmap-ul:**
- Fiecare rând e o moleculă, fiecare coloană un descriptor.
- Rândul **0** = afinitate cea mai mică (legare slabă); rândul **200** = afinitate cea mai mare.
- Roșu = valoare mare a featurului față de medie; albastru = valoare mică.

Caută **gradiente verticale** — coloane care variază sistematic de sus în jos. Dacă există, acel descriptor e predictiv (corelat liniar cu target-ul). Dacă heatmap-ul arată ca zgomot pur — nu sunt features ușor de exploatat liniar.

## Cap. 5 — Modele de predicție

Avem ~$(round(nrow(df_feat)/1000, digits=1))K complexe, fiecare cu 9 features și un target `value`. Antrenăm două modele:

### 1. Baseline: regresie liniară
$$\hat{y} = \mathbf{w}^\top \mathbf{x} + b$$
Cea mai simplă predicție posibilă. **Dacă MLP-ul e mai prost decât asta**, ceva e fundamental greșit (suprafit, bug, sau task imposibil cu features-urile date).

### 2. MLP cu Flux.jl
Arhitectură: $9 \to 32 \to 16 \to 1$ cu activări ReLU. Aceeași Flux ca-n notebook 03, dar pe date reale.

### Train / val / test
Folosim **split-ul oficial LP-PDBBind**:
- **train** — antrenarea greutăților;
- **val** — alegerea hyperparameter-ilor (rata de învățare, număr epoci);
- **test** — raportare finală, **niciodată folosit la decizii**.

### Standardizarea features-urilor
Standardizăm fiecare coloană la $\mathcal{N}(0, 1)$ folosind **media și std din train** — niciodată din val sau test. Asta previne *information leakage*.

In [ ]:
df_train = filter(row -> row.new_split == "train", df_feat)
df_val   = filter(row -> row.new_split == "val",   df_feat)
df_test  = filter(row -> row.new_split == "test",  df_feat)

println("Train: $(nrow(df_train))")
println("Val:   $(nrow(df_val))")
println("Test:  $(nrow(df_test))")

X_train = Matrix(df_train[!, feature_cols])
X_val   = Matrix(df_val[!,   feature_cols])
X_test  = Matrix(df_test[!,  feature_cols])
y_train = Float64.(df_train.value)
y_val   = Float64.(df_val.value)
y_test  = Float64.(df_test.value)

# Standardizare cu μ,σ DIN TRAIN
μ = mean(X_train, dims=1)
σ = std(X_train, dims=1)

X_train_n = (X_train .- μ) ./ σ
X_val_n   = (X_val   .- μ) ./ σ
X_test_n  = (X_test  .- μ) ./ σ

println("\n✓ Date pregătite și standardizate.")
println("  μ_train = ", round.(vec(μ), digits=2))
println("  σ_train = ", round.(vec(σ), digits=2))

### 5.1 Baseline — regresie liniară cu GLM.jl

`GLM.jl` e pachetul standard în Julia pentru *Generalized Linear Models*. Folosim funcția `lm` (linear model) cu formulă explicită.

In [ ]:
df_train_glm = DataFrame(X_train_n, string.(feature_cols))
df_train_glm.y = y_train

model_lin = lm(@formula(y ~ mw + logp + hbd + hba + rotors + heavy + rings + arom_rings + resolution),
               df_train_glm)

println(model_lin)

**Citirea tabelului GLM:**
- `Coef.` = coeficienții (greutățile $w_i$).
- `Std. Error` = incertitudinea estimării (mai mic = mai sigur).
- `t` și `Pr(>|t|)` = test de semnificație: `Pr(>|t|) < 0.05` înseamnă feature-ul *probabil* are efect.
- `(Intercept)` ≈ media lui $y$ (după standardizare).

Coeficienții arată **direcția** efectului fiecărui feature (pozitiv / negativ) și **magnitudinea** (cu cât crește predicția când feature-ul crește cu 1 deviație standard).

In [ ]:
function metrics(y_true, y_pred)
    mae = mean(abs.(y_true .- y_pred))
    rmse = sqrt(mean((y_true .- y_pred).^2))
    ss_res = sum((y_true .- y_pred).^2)
    ss_tot = sum((y_true .- mean(y_true)).^2)
    r2 = 1 - ss_res / ss_tot
    return (mae=mae, rmse=rmse, r2=r2)
end

df_test_glm = DataFrame(X_test_n, string.(feature_cols))
y_pred_lin = predict(model_lin, df_test_glm)

m_lin = metrics(y_test, y_pred_lin)
println("📈 Baseline liniar pe test:")
println("   MAE  = $(round(m_lin.mae, digits=3))")
println("   RMSE = $(round(m_lin.rmse, digits=3))")
println("   R²   = $(round(m_lin.r2, digits=3))")

**Interpretare:** un baseline R² de 0.1-0.3 e tipic pentru featurizare naivă pe afinități proteină-ligand. Features-urile noastre explică ~10-30% din varianța afinităților — restul e structură 3D, dinamică, efecte electronice fine pe care nu le surprindem.

### 5.2 MLP cu Flux.jl

In [ ]:
function build_mlp(input_dim=9, hidden=[32, 16])
    layers = []
    in_dim = input_dim
    for h in hidden
        push!(layers, Dense(in_dim => h, relu))
        in_dim = h
    end
    push!(layers, Dense(in_dim => 1))
    return Chain(layers...)
end

Random.seed!(42)
model = build_mlp()
println(model)

n_params = sum(length, Flux.params(model))
println("\nParametri totali: $n_params")

In [ ]:
# Funcție de loss
loss(model, x, y) = Flux.mse(vec(model(x)), y)

# Convertim la Float32 (Flux preferă single precision)
X_train_f = Float32.(X_train_n')   # (features × samples)
X_val_f   = Float32.(X_val_n')
X_test_f  = Float32.(X_test_n')
y_train_f = Float32.(y_train)
y_val_f   = Float32.(y_val)
y_test_f  = Float32.(y_test)

# Hyperparameters
epochs = 100
batch_size = 64
lr = 0.001

opt_state = Flux.setup(Adam(lr), model)

train_losses = Float64[]
val_losses = Float64[]
n_train = size(X_train_f, 2)

println("⏳ Antrenez MLP pentru $epochs epochi...")
for epoch in 1:epochs
    idx = shuffle(1:n_train)
    for i in 1:batch_size:n_train
        batch_idx = idx[i:min(i+batch_size-1, n_train)]
        x_batch = X_train_f[:, batch_idx]
        y_batch = y_train_f[batch_idx]
        Flux.train!(loss, model, [(x_batch, y_batch)], opt_state)
    end

    push!(train_losses, loss(model, X_train_f, y_train_f))
    push!(val_losses,   loss(model, X_val_f,   y_val_f))

    if epoch % 10 == 0
        println("  Epoch $epoch: train=$(round(train_losses[end], digits=3)), val=$(round(val_losses[end], digits=3))")
    end
end
println("✓ Antrenat.")

In [ ]:
plot(1:epochs, train_losses, label="Train", lw=2, color=:steelblue,
     xlabel="Epoch", ylabel="Loss (MSE)",
     title="Curba de antrenament MLP", size=(750, 420))
plot!(1:epochs, val_losses, label="Validation", lw=2, color=:orange)

**Diagnostic:**
- Dacă `val_loss` crește în timp ce `train_loss` scade → **overfitting**. Adaugă regularization, dropout, sau early stopping.
- Dacă ambele scad și se stabilizează → modelul a învățat tot ce poate.
- Dacă ambele rămân mari → modelul e prea simplu sau task-ul e prea greu cu features-urile date.

### 5.3 Comparație finală

In [ ]:
y_pred_mlp = vec(model(X_test_f))
m_mlp = metrics(y_test_f, y_pred_mlp)

println("📈 MLP Flux pe test:")
println("   MAE  = $(round(m_mlp.mae, digits=3))")
println("   RMSE = $(round(m_mlp.rmse, digits=3))")
println("   R²   = $(round(m_mlp.r2, digits=3))")

println("\n🆚 Comparație directă:")
println("            |  MAE   |  RMSE  |   R²")
println("            +--------+--------+---------")
println("Linear      |  $(round(m_lin.mae, digits=3))  |  $(round(m_lin.rmse, digits=3))  |  $(round(m_lin.r2, digits=3))")
println("MLP (Flux)  |  $(round(m_mlp.mae, digits=3))  |  $(round(m_mlp.rmse, digits=3))  |  $(round(m_mlp.r2, digits=3))")

improvement = round(100 * (m_mlp.r2 - m_lin.r2) / max(abs(m_lin.r2), 0.01), digits=1)
println("\nÎmbunătățire R² față de baseline: $improvement%")

### 5.4 Parity plots — predicție vs. real

Un *parity plot* (Y vs. Ŷ) e cea mai informativă vizualizare pentru regresie. Linia roșie e $y = \hat{y}$ — predicția perfectă.

In [ ]:
ax_min, ax_max = 2, 12

p_lin = scatter(y_test, y_pred_lin, alpha=0.4, markersize=3, color=:steelblue,
    xlabel="pKd/pKi real", ylabel="pKd/pKi prezis",
    title="Linear baseline (R²=$(round(m_lin.r2, digits=2)))",
    legend=false, aspect_ratio=:equal,
    xlims=(ax_min, ax_max), ylims=(ax_min, ax_max))
plot!(p_lin, [ax_min, ax_max], [ax_min, ax_max], color=:red, lw=2, linestyle=:dash)

p_mlp = scatter(y_test_f, y_pred_mlp, alpha=0.4, markersize=3, color=:purple,
    xlabel="pKd/pKi real", ylabel="pKd/pKi prezis",
    title="MLP Flux (R²=$(round(m_mlp.r2, digits=2)))",
    legend=false, aspect_ratio=:equal,
    xlims=(ax_min, ax_max), ylims=(ax_min, ax_max))
plot!(p_mlp, [ax_min, ax_max], [ax_min, ax_max], color=:red, lw=2, linestyle=:dash)

plot(p_lin, p_mlp, layout=(1, 2), size=(1000, 500))

**Cum citești un parity plot:**
- Punctele aproape de linia roșie = predicții bune.
- *Compression spre mijloc* (modelul prezice 5–7 indiferent de target real) = model "leneș" — nu generalizează la extreme.
- *Spread vertical mare la o valoare X dată* = featurizarea nu deosebește moleculele cu același feature dar afinități diferite.

### 5.5 Reziduuri

In [ ]:
residuals_mlp = y_test_f .- y_pred_mlp

p1 = scatter(y_pred_mlp, residuals_mlp, alpha=0.4, markersize=3,
    xlabel="Predicție MLP", ylabel="Reziduu (real - prezis)",
    title="Reziduuri vs. predicție", legend=false, color=:purple)
hline!(p1, [0], color=:red, lw=2)

p2 = histogram(residuals_mlp, bins=40,
    xlabel="Reziduu", ylabel="Număr",
    title="Distribuția reziduurilor",
    color=:purple, alpha=0.7, legend=false)
vline!(p2, [0], color=:red, lw=2)

plot(p1, p2, layout=(1, 2), size=(1000, 420))

println("Reziduuri MLP pe test:")
println("  Media: $(round(mean(residuals_mlp), digits=4))")
println("  Std:   $(round(std(residuals_mlp), digits=3))")
println("  Median: $(round(median(residuals_mlp), digits=4))")

**Interpretare reziduuri:**
- **Media ≈ 0** ✓ — modelul nu e bias-uit sistematic într-o direcție.
- **Distribuție aproape normală** — assumption-ul de bază al modelelor de regresie (nu strict necesar pentru MLP, dar e un semn bun).
- **Niciun pattern în scatter** — modelul a "stors" tot semnalul liniar disponibil. Dacă ai un pattern, însemnă că ai mai multă semnalitate de extras (poate cu features-uri noi sau model mai complex).

## Cap. 6 — Vizualizare 3D pe exemple concrete

Numerele globale (R², MAE) sunt impresia "de pe orbită". Hai să ne uităm la **molecule individuale** — un complex prezis foarte bine vs. unul ratat. Asta ne dă intuiție despre **când și de ce eșuează** modelul.

In [ ]:
df_test_results = copy(df_test)
df_test_results.y_pred = y_pred_mlp
df_test_results.residual_abs = abs.(y_test_f .- y_pred_mlp)

# Top 5 cele mai bune
sort!(df_test_results, :residual_abs)
println("🎯 TOP 5 PREDICȚII FOARTE BUNE:")
for row in eachrow(first(df_test_results, 5))
    println("  PDB $(row.header) ($(row.type))")
    println("    real=$(round(row.value, digits=2)), prezis=$(round(row.y_pred, digits=2)), |eroare|=$(round(row.residual_abs, digits=3))")
end

println()
println("💥 TOP 5 PREDICȚII FOARTE PROASTE:")
sort!(df_test_results, :residual_abs, rev=true)
for row in eachrow(first(df_test_results, 5))
    println("  PDB $(row.header) ($(row.type))")
    println("    real=$(round(row.value, digits=2)), prezis=$(round(row.y_pred, digits=2)), |eroare|=$(round(row.residual_abs, digits=3))")
end

### Vizualizare 3D — un hit și un miss

Folosim `Bio3DView.jl` (deja folosit în notebook 04) ca să descărcăm structurile din RCSB PDB și să le afișăm interactiv în notebook.

In [ ]:
sort!(df_test_results, :residual_abs)
best_pdb  = uppercase(df_test_results.header[1])
sort!(df_test_results, :residual_abs, rev=true)
worst_pdb = uppercase(df_test_results.header[1])

println("🎯 Cea mai bună predicție:  PDB $best_pdb")
println("💥 Cea mai proastă predicție: PDB $worst_pdb")

**Cea mai bună predicție** — un complex unde features-urile noastre captează bine afinitatea:

In [ ]:
HTML(viewpdb(best_pdb;
             style=Style("cartoon", Dict("color" => "spectrum")),
             html=true))

**Cea mai proastă predicție** — un complex unde features-urile naive nu surprind ce face afinitatea atât de mare/mică:

In [ ]:
HTML(viewpdb(worst_pdb;
             style=Style("cartoon", Dict("color" => "spectrum")),
             html=true))

**Observă:** ambele structuri pot să arate "la fel" la nivel cartoon — același tip de fold, dimensiuni comparabile. Ce **diferă** sunt detaliile interfeței proteină-ligand pe care features-urile noastre 1D nu le văd:
- forma exactă a buzunarului de legare,
- aranjamentul 3D al reziduurilor cheie,
- prezența/absența unei legături H specifice,
- interacțiuni de tip cation-π, π-π, halogen bond.

**Aici intră ML-ul geometric** (GNN, EquiBind, DiffDock) — modelele care iau coordonate 3D ca input.

## Cap. 7 — Discuție și limitări

### 7.1 De ce nu mai bun?

Modelul nostru obține un R² modest (de obicei 0.2–0.4 pe LP-PDBBind cu features 1D). De ce nu mai bine?

**1. Featurizare insuficientă.** Folosim 9 descriptori 1D ai ligandului. Asta **ignoră complet structura 3D a proteinei**! Modelele state-of-the-art:
- folosesc graful molecular ca input direct (Graph Neural Networks);
- includ secvența proteinei prin embeddings ESM-2 [Lin et al., 2023];
- modelează interfața de legare 3D (proteină-ligand contacte) cu rețele *echivariante* [Stärk et al., 2022; Corso et al., 2023];
- ajung la R² de 0.5–0.7 pe LP-PDBBind.

**2. Features-uri de proteină lipsesc.** Nu folosim secvența `seq`! Două proteine foarte diferite cu același ligand vor primi exact aceeași predicție. Asta e o **problemă structurală** a abordării, nu un detaliu de tuning.

**3. Bias-ul dataset-ului.** PDBbind e dominat de hidrolaze, kinaze, proteaze. Pentru ținte rare (GPCR-uri, canale ionice), modelul nostru va eșua chiar mai mult decât R²-ul global sugerează.

### 7.2 De unde intră HGNN (notebook 04) în joc?

Hypergraph Neural Networks din nb 04 sunt o opțiune **promițătoare** pentru această sarcină. De ce?

Interfața proteină-ligand are **interacțiuni de ordin înalt** (nu doar perechi):
- un *buzunar de legare* e un **set** de 5–15 reziduuri care colaborează (nu o secvență de perechi);
- legăturile H, contactele hidrofobe și efectele π-π formează "rețele" de constrângeri simultane;
- modelele HGNN pot modela natural aceste interacțiuni *n-ary* — vezi [Wang et al., 2023] pentru o aplicație DTI cu hypergraphuri.

### 7.3 Pași imediați pentru îmbunătățire

Dacă vrei să continui pe acest dataset, în ordinea crescătoare a câștigului:

| Pas | Effort | Câștig probabil R² |
|---|---|---|
| Adaugă **Morgan fingerprints (ECFP)** ca features | Mic — `morgan_fp_vector(mol, 2, 2048)` din MolecularGraph.jl | 0.10–0.15 |
| Folosește **Random Forest** pe features extinse | Mic — `MLJ.jl` sau `DecisionTree.jl` | 0.10–0.15 |
| **GNN** simplu (GraphNeuralNetworks.jl) | Mediu | 0.10–0.20 |
| Embeddings **ESM-2** pentru proteine (PyCall + esm) | Mediu | 0.15–0.25 |
| **HGNN** pe graf proteină-ligand | Mare (research-grade) | 0.20–0.35 |

### 7.4 Handoff la UCD

La interviul de la UCD, întreabă coordonatorul:
1. **Ce reprezentare folosesc?** SMILES / graph molecular 2D / coordonate 3D / ESM-2?
2. **Care e benchmark-ul?** PDBbind, BindingDB, ChEMBL, DAVIS, KIBA?
3. **Ce modele baseline?** Random Forest pe Morgan FP, GNN simplu, EquiBind?
4. **Compute?** GPU local sau cluster (Sonic la UCD)?
5. **Limbaj?** Julia (Flux + GraphNeuralNetworks) sau Python (PyTorch + PyG)?
6. **Task exact?** Regresie (afinitate) sau clasificare (active/inactive)?

## Ce ai învățat în acest notebook

- ✅ **Pipeline ML end-to-end real** — download → curățare → EDA → featurizare → train → eval → vizualizare.
- ✅ **Analiză exploratorie disciplinată** — distribuții, statistici, corelații, vizualizări. *Niciodată* să nu sari peste EDA.
- ✅ **Featurizare moleculară** cu `MolecularGraph.jl` — descriptori Lipinski-style + ce înseamnă fiecare.
- ✅ **Baseline first, complex second** — întotdeauna compară un MLP cu o regresie liniară simplă. Dacă nu e mai bun, ceva e greșit.
- ✅ **Split leak-proof** — de ce contează **cum** împarți datele, nu doar **cât** de bine antrenezi.
- ✅ **Metrici de regresie** — MAE, RMSE, R² și cum se citesc parity / residual plots.
- ✅ **Limitări sincere** — de unde vine plafonul cu features 1D și cum HGNN/GNN/ESM pot ridica plafonul.

### Concluzia importantă

În ML aplicat la chimie/biologie, **alegerea reprezentării contează mai mult decât arhitectura modelului**. Un MLP simplu pe ESM-2 embeddings va bate ușor un transformer complicat pe descriptori 1D Lipinski. Asta vei vedea direct la UCD: primul lucru pe care îl întrebi când vezi un paper sau un proiect este: **"Ce reprezintă input-ul, exact?"**

## Referințe

### Dataset
- **Li, J., Guan, X., Zhang, O., et al. (2023).** Leak Proof PDBBind: A Reorganized Dataset of Protein-Ligand Complexes for More Generalizable Binding Affinity Prediction. *Journal of Chemical Information and Modeling*. [arXiv:2308.09639](https://arxiv.org/abs/2308.09639) — **sursa noastră de date**.
- **Liu, Z., Su, M., Han, L., et al. (2017).** Forging the Basis for Developing Protein-Ligand Interaction Scoring Functions. *Accounts of Chemical Research*, 50(2), 302–309. [DOI:10.1021/acs.accounts.6b00491](https://doi.org/10.1021/acs.accounts.6b00491) — paper-ul original PDBbind.
- **Wang, R., Fang, X., Lu, Y., & Wang, S. (2004).** The PDBbind Database: Collection of Binding Affinities for Protein-Ligand Complexes with Known Three-Dimensional Structures. *J. Med. Chem.*, 47(12), 2977–2980. [DOI:10.1021/jm030580l](https://doi.org/10.1021/jm030580l)

### Featurizare moleculară
- **Lipinski, C. A., Lombardo, F., Dominy, B. W., & Feeney, P. J. (2001).** Experimental and computational approaches to estimate solubility and permeability in drug discovery and development settings. *Advanced Drug Delivery Reviews*, 46(1–3), 3–26. [DOI:10.1016/S0169-409X(00)00129-0](https://doi.org/10.1016/S0169-409X(00)00129-0) — *Lipinski's Rule of 5*.
- **Wildman, S. A., & Crippen, G. M. (1999).** Prediction of Physicochemical Parameters by Atomic Contributions. *J. Chem. Inf. Comput. Sci.*, 39(5), 868–873. [DOI:10.1021/ci990307l](https://doi.org/10.1021/ci990307l) — *LogP estimator*.
- **Rogers, D., & Hahn, M. (2010).** Extended-Connectivity Fingerprints. *J. Chem. Inf. Model.*, 50(5), 742–754. [DOI:10.1021/ci100050t](https://doi.org/10.1021/ci100050t) — *Morgan / ECFP*.

### Modele state-of-the-art pentru afinitate
- **Stärk, H., Ganea, O., Pattanaik, L., Barzilay, R., & Jaakkola, T. (2022).** EquiBind: Geometric Deep Learning for Drug Binding Structure Prediction. *ICML 2022*. [arXiv:2202.05146](https://arxiv.org/abs/2202.05146)
- **Corso, G., Stärk, H., Jing, B., Barzilay, R., & Jaakkola, T. (2023).** DiffDock: Diffusion Steps, Twists, and Turns for Molecular Docking. *ICLR 2023*. [arXiv:2210.01776](https://arxiv.org/abs/2210.01776)
- **Lin, Z., Akin, H., Rao, R., et al. (2023).** Evolutionary-scale prediction of atomic-level protein structure with a language model. *Science*, 379(6637), 1123–1130. [DOI:10.1126/science.ade2574](https://doi.org/10.1126/science.ade2574) — *ESM-2*.
- **Wang, T., Liu, Y., Yin, Q., et al. (2023).** HyperAttentionDTI: improving drug-protein interaction prediction by sequence-based deep learning with attention mechanism. *Bioinformatics*. — *DTI cu hypergraphuri / attention*.

### Statistică
- **Cox, D. R. (2005).** Frequentist statistics as a theory of inductive inference. *Lehmann Symposium Optimality*, IMS Lecture Notes—Monograph Series, Vol. 49. [Project Euclid](https://projecteuclid.org/journals/lecture-notes-monograph-series).

### Pachete Julia folosite
- [**MolecularGraph.jl**](https://github.com/mojaie/MolecularGraph.jl) — Mojaie M. & contribuitori. SMILES parsing + descriptori. Documentație: <https://mojaie.github.io/MolecularGraph.jl/stable/>.
- [**Flux.jl**](https://fluxml.ai/Flux.jl/) — Innes, M. (2018). Flux: Elegant Machine Learning with Julia. *Journal of Open Source Software*, 3(25), 602. [DOI:10.21105/joss.00602](https://doi.org/10.21105/joss.00602)
- [**GLM.jl**](https://github.com/JuliaStats/GLM.jl) — JuliaStats. Modele liniare generalizate.
- [**Bio3DView.jl**](https://github.com/jgreener64/Bio3DView.jl) — Greener, J. G. (2018). Wrapper Julia pentru 3Dmol.js.
- [**DataFrames.jl**](https://github.com/JuliaData/DataFrames.jl) — Bouchet-Valat, M., & Kamiński, B. (2023). *Journal of Statistical Software*, 107(4). [DOI:10.18637/jss.v107.i04](https://doi.org/10.18637/jss.v107.i04)
- [**CSV.jl**](https://github.com/JuliaData/CSV.jl) — Quinn, J. și contribuitori.
- [**Plots.jl**](https://github.com/JuliaPlots/Plots.jl) — Christ, S., et al. (2023). [arXiv:2204.08775](https://arxiv.org/abs/2204.08775).

### Cărți / cursuri pentru aprofundare
- **Hamilton, W. L. (2020).** *Graph Representation Learning*. Morgan & Claypool. <https://www.cs.mcgill.ca/~wlh/grl_book/>
- **Murphy, K. (2022).** *Probabilistic Machine Learning: An Introduction*. MIT Press. <https://probml.github.io/pml-book/>
- **Stanford CS224W** — Machine Learning with Graphs (Leskovec). <https://web.stanford.edu/class/cs224w/>
- **Deep Learning for Drug Discovery and Biomedicine** (curs ETH Zurich, Krause/Stark). <https://las.inf.ethz.ch/teaching/dlb-fs23>